# Agente Explicador de Regras do Futebol com RAG

## Carregamento de bibliotecas

In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv
import os

In [14]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("Chave carregada:", api_key[:5] + "...")

Chave carregada: sk-or...


## Carregamento dos Documentos

In [4]:
caminho_pdf = './data/Regras_do_Jogo_2023_24.pdf'

loader = PyPDFLoader(caminho_pdf)
documents = loader.load()

len(documents)

232

## Preparação dos Documentos

In [10]:
# Divisão do documentos em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

len(chunks)

654

## Criação dos Embeddings e armazenamento no banco vetorial

In [15]:
# criação dos embeddings
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1"
)

In [17]:
# criação do banco vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_regras_futebol'
)

## Recuperação de Contexto (Retriever)

In [18]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 3}
)

## Integração com LLM

In [19]:
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

In [27]:
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'Responda usando apenas o contexto.'),
        ('human', "Pergunta: {question}\n\nContexto:\n{context}")
    ]
)

# construção da pipeline
chain  = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x['question']
    }
    | prompt
    |llm
)

full_chain = RunnableParallel(
    answer=chain,
    context = lambda x: retriever.invoke(x['question'])
)

## Testes e validação

In [32]:
pergunta = "Um jogador pode usar a mão para marcar um gol?"

response = full_chain.invoke({"question": pergunta})

print(f"Pergunta: {pergunta}")
print(f"\nResposta da LLM: {response['answer'].content}")

print("\nDocumentos Utilizados como contexto:")
idx = 1
for doc in response["context"]:
    print(f"{idx}) {doc.page_content[:200]}\n****Fonte:{doc.metadata.get("source", "Documento desconhecimento")}\n****Página: {doc.metadata.get('page', 'N/A')}")
    idx += 1

Pergunta: Um jogador pode usar a mão para marcar um gol?

Resposta da LLM: Sim, um jogador pode usar a mão para marcar um gol, desde que o movimento da mão ou do braço seja uma consequência do movimento do corpo do jogador nessa ação específica e possa ser justificado por esse movimento.

Documentos Utilizados como contexto:
1) de forma antinatural quando a posição de sua mão ou seu braço não for 
consequência do movimento do corpo nessa ação específica ou não puder ser 
justificada por esse movimento. Ao colocar a mão ou o 
****Fonte:./data/Regras_do_Jogo_2023_24.pdf
****Página: 101
2) (com ou sem a mão ou o braço) após o reinício do jogo e antes que ela toque em 
outro jogador, o goleiro deve ser sancionado se a infração interromper um ataque 
promissor ou impedir um adversário ou 
****Fonte:./data/Regras_do_Jogo_2023_24.pdf
****Página: 102
3) todos os contatos da mão ou do braço de um jogador com a bola constituem 
uma infração.
No entanto, cometerá uma infração o jogador que: 
•  t

In [33]:
pergunta = "O que caracteriza a posição de impedimento no futebol?"

response = full_chain.invoke({"question": pergunta})

print(f"Pergunta: {pergunta}")
print(f"\nResposta da LLM: {response['answer'].content}")

print("\nDocumentos Utilizados como contexto:")
idx = 1
for doc in response["context"]:
    print(f"{idx}) {doc.page_content[:200]}\n****Fonte:{doc.metadata.get("source", "Documento desconhecimento")}\n****Página: {doc.metadata.get('page', 'N/A')}")
    idx += 1

Pergunta: O que caracteriza a posição de impedimento no futebol?

Resposta da LLM: Uma posição de impedimento acontece quando a cabeça, o corpo ou os pés do jogador estão entrando em uma área específica no campo do adversário, que é definida pela linha de fundo. Isso significa que o jogador está entrando em uma área que é considerada pertencente ao adversário. Não há infração, somente mesmo que o jogador esteja em posição de impedimento. No entanto, se ele for objeto de uma falta e comprometer a capacidade de um adversário de tocar na bola, será considerado infração por impedimento.

Documentos Utilizados como contexto:
1) 95
11
Regras do Jogo 2023/24  |  Regra 11  |  O impedimento
1. Posição de impedimento
Encontrar-se em posição de impedimento não é uma infração.
Um jogador estará em posição de impedimento se:
• 
 qua
****Fonte:./data/Regras_do_Jogo_2023_24.pdf
****Página: 94
2) isto será considerado infração por impedimento se afetar a capacidade do 
adversário de tocar na bola ou d